# 01 — Veri kontrolü, bölünme ve keşifsel veri analizi (EDA)

Bu notebook şu adımları sırasıyla yapar:

1. Ham `bank-additional.csv` dosyasını okur; değişken adlarını, veri tiplerini ve hedef etiketleri kontrol eder.
2. Tamamen aynı (kopya) kayıtları inceler.
3. Veriyi stratified biçimde %80 development ve %20 bağımsız test olarak ayırır ve `results/dataset_split.csv` dosyasını yazar.
4. EDA'yı **yalnızca development kümesinde** yapar. Bağımsız test kümesinin özellikleri ve etiketleri bu bölümde hiç kullanılmaz.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import project as P
P.set_global_seeds()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
print("Temel seed:", P.BASE_SEED); print("Alt seedler:", P.SEEDS)

Temel seed: 31074
Alt seedler: {'python': 31074, 'numpy': 31075, 'torch': 31076, 'train_test_split': 31077, 'outer_cv': 31078, 'inner_cv': 31079, 'early_stop_split': 31080, 'random_search': 31081, 'bootstrap': 31082, 'torch_stability_2': 31176, 'torch_stability_3': 31276}


## 1. Ham verinin okunması ve tip/etiket kontrolü

In [2]:
raw = P.load_raw()
print("Boyut:", raw.shape)
print("Sütunlar:", list(raw.columns))
print(raw.dtypes.to_string())
print("\nHedef etiketleri:", raw["y"].value_counts().to_dict())
print("Gerçek NaN sayısı (tüm veri):", int(raw.isna().sum().sum()))
assert raw.shape == (4119, 21)
assert set(raw["y"].unique()) == {"yes", "no"}
expected = set(P.NUMERIC_COLS + P.CATEGORICAL_COLS + ["duration", "y"])
assert set(raw.columns) == expected, set(raw.columns) ^ expected

Boyut: (4119, 21)
Sütunlar: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str

Hedef etiketleri: {'no': 3668, 'yes': 451}
Gerçek NaN sayısı (tüm veri): 0


## 2. Kopya kayıt kontrolü

Kontrol iki düzeyde yapılır: (a) 21 sütunun tamamı aynı olan kayıtlar, (b) `duration` çıkarıldıktan sonra kalan 19 girdi özelliği aynı olan kayıtlar. Bu kontrol hedefle ilişki kurmaz; yalnızca veri kalitesi kontrolüdür.

In [3]:
dup_all = raw.duplicated(keep=False)
print("(a) 21 sütunu tamamen aynı kayıt sayısı:", int(dup_all.sum()))
feat_cols = [c for c in raw.columns if c not in ("duration", "y")]
dup_feat = raw.duplicated(subset=feat_cols, keep=False)
print("(b) duration çıkarıldıktan sonra özellikleri aynı kayıt sayısı:", int(dup_feat.sum()))
dup_groups = raw[dup_feat].groupby(feat_cols, dropna=False).ngroup()
dup_table = pd.DataFrame({"original_index": raw.index[dup_feat], "dup_group": dup_groups.values})
print("Grup sayısı:", dup_table["dup_group"].nunique(),
      "| grup büyüklükleri:", dup_table.groupby("dup_group").size().value_counts().to_dict())

(a) 21 sütunu tamamen aynı kayıt sayısı: 0
(b) duration çıkarıldıktan sonra özellikleri aynı kayıt sayısı: 40
Grup sayısı: 20 | grup büyüklükleri: {2: 20}


**Uygulanan kural:** Ham dosyada 21 sütunu birebir aynı olan kayıt yoktur. Aynı özellik vektörüne sahip çiftler yalnızca zorunlu `duration` çıkarımından sonra ortaya çıkar; bu satırlar orijinal dosyada görüşme süresiyle birbirinden ayrılan farklı kayıtlardır. Bu nedenle hiçbir satır silinmez. Ödev bölünmeler için `StratifiedKFold` ve stratified split istediği için grup tabanlı bölme kullanılmaz. Bunun yerine bu çiftlerin bölünmelerde hangi tarafa düştüğü aşağıda sayılarak raporlanır. Bu durum, aynı özellik vektörünün hem eğitim hem değerlendirme tarafında bulunabileceği küçük bir sızıntı kaynağıdır ve raporda sınırlama olarak belirtilir.

## 3. Stratified %80 / %20 bölünme

In [4]:
X_all, y_all = P.prepare_xy(raw)
X_dev, X_test, y_dev, y_test = P.make_train_test_split(X_all, y_all)
rows = []
for name, yy in [("development", y_dev), ("test", y_test)]:
    for i in yy.index:
        rows.append((int(i), name))
split = pd.DataFrame(rows, columns=["original_index", "subset"]).sort_values("original_index")
summary = pd.DataFrame({
    "subset": ["development", "test", "all"],
    "n": [len(y_dev), len(y_test), len(y_all)],
    "n_positive": [int(y_dev.sum()), int(y_test.sum()), int(y_all.sum())],
})
summary["n_negative"] = summary["n"] - summary["n_positive"]
summary["positive_pct"] = 100 * summary["n_positive"] / summary["n"]
summary["negative_pct"] = 100 * summary["n_negative"] / summary["n"]
split = split.merge(summary[["subset", "n", "n_positive", "positive_pct", "negative_pct"]]
                    .rename(columns={"n": "subset_n", "n_positive": "subset_n_positive",
                                     "positive_pct": "subset_positive_pct",
                                     "negative_pct": "subset_negative_pct"}), on="subset")
split["split_seed"] = P.SEEDS["train_test_split"]
(P.RESULTS).mkdir(exist_ok=True)
split.to_csv(P.RESULTS / "dataset_split.csv", index=False)
summary.to_csv(P.RESULTS / "dataset_split_summary.csv", index=False)
summary

,subset,n,n_positive,n_negative,positive_pct,negative_pct
0,development,3295,361,2934,10.955994,89.044006
1,test,824,90,734,10.922330,89.077670
2,all,4119,451,3668,10.949260,89.050740


In [5]:
dup_table["subset"] = dup_table["original_index"].map(split.set_index("original_index")["subset"])
cross = dup_table.groupby("dup_group")["subset"].nunique()
print("Dev ile test arasında bölünen kopya-özellik grubu sayısı:", int((cross > 1).sum()), "/", len(cross))
dup_table.to_csv(P.RESULTS / "duplicate_feature_groups.csv", index=False)

Dev ile test arasında bölünen kopya-özellik grubu sayısı: 8 / 20


## 4. EDA (yalnızca development kümesi)

Bu noktadan sonra yalnızca `X_dev` ve `y_dev` kullanılır. EDA için `pdays` değişkeni ham hâliyle (999 dahil) incelenir; 999 değeri ayrı bir özel kod olarak sayılır.

In [6]:
dev = raw.loc[X_dev.index].drop(columns=["duration"]).copy()
dev["y"] = y_dev.values
del X_test, y_test  # test kümesi bu notebook'ta bundan sonra kullanılmaz
print("Development boyutu:", dev.shape)

Development boyutu: (3295, 20)


### 4.1 Özellik türleri, benzersiz değerler, eksik ve özel kodlu değerler (Tablo 1a)

In [7]:
feat = [c for c in dev.columns if c != "y"]
t1 = pd.DataFrame({
    "type": ["sayısal" if c in P.NUMERIC_COLS else "kategorik" for c in feat],
    "dtype": [str(dev[c].dtype) for c in feat],
    "n_unique": [dev[c].nunique() for c in feat],
    "n_NaN": [int(dev[c].isna().sum()) for c in feat],
    "n_unknown": [int((dev[c] == "unknown").sum()) if c in P.CATEGORICAL_COLS else 0 for c in feat],
}, index=feat)
t1["pct_unknown"] = 100 * t1["n_unknown"] / len(dev)
t1.loc["pdays", "special_code"] = f"999 (daha önce temas yok): {(dev['pdays'] == 999).sum()} satır, %{100*(dev['pdays'] == 999).mean():.2f}"
t1["special_code"] = t1["special_code"].fillna("")
t1.index.name = "feature"
t1.to_csv(P.RESULTS / "eda_feature_overview.csv")
print("Girdi özelliği sayısı (duration hariç):", len(feat), "| sayısal:", (t1.type == "sayısal").sum(),
      "| kategorik:", (t1.type == "kategorik").sum())
print("Development içinde 19 özelliği aynı satır sayısı:", int(dev[feat].duplicated(keep=False).sum()))
t1

Girdi özelliği sayısı (duration hariç): 19 | sayısal: 9 | kategorik: 10
Development içinde 19 özelliği aynı satır sayısı: 24


,type,dtype,n_unique,n_NaN,n_unknown,pct_unknown,special_code
feature,,,,,,,
age,sayısal,int64,65,0,0,0.000000,
job,kategorik,str,12,0,29,0.880121,
marital,kategorik,str,4,0,6,0.182094,
education,kategorik,str,7,0,134,4.066768,
default,kategorik,str,3,0,650,19.726859,
housing,kategorik,str,3,0,83,2.518968,
loan,kategorik,str,3,0,83,2.518968,
contact,kategorik,str,2,0,0,0.000000,
month,kategorik,str,10,0,0,0.000000,


### 4.2 Sınıf dağılımı

In [8]:
cls = dev["y"].value_counts().rename({0: "no (0)", 1: "yes (1)"}).to_frame("n")
cls["pct"] = 100 * cls["n"] / cls["n"].sum()
cls.to_csv(P.RESULTS / "eda_class_distribution.csv")
print(cls)
fig, ax = plt.subplots(figsize=(4.5, 3.2))
bars = ax.bar(cls.index, cls["n"], color=["#7f8c9a", "#2b6cb0"])
for b, (n, p) in zip(bars, zip(cls["n"], cls["pct"])):
    ax.text(b.get_x() + b.get_width()/2, n, f"{n}\n(%{p:.1f})", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Gözlem sayısı"); ax.set_title("Development sınıf dağılımı")
ax.set_ylim(0, cls["n"].max() * 1.2)
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_eda_class_distribution.png", dpi=200); plt.close(fig)

            n        pct
y                       
no (0)   2934  89.044006
yes (1)   361  10.955994


### 4.3 Sayısal özelliklerin tanımlayıcı istatistikleri (Tablo 1b), histogram ve boxplot

In [9]:
num = dev[P.NUMERIC_COLS].astype(float)
def describe(s):
    q1, q3 = s.quantile([.25, .75])
    return pd.Series({"n": s.notna().sum(), "mean": s.mean(), "median": s.median(), "SD": s.std(ddof=1),
                      "Q1": q1, "Q3": q3, "IQR": q3 - q1, "min": s.min(), "max": s.max()})
t2 = num.apply(describe).T
pd_real = num["pdays"][num["pdays"] != 999]
t2.loc["pdays (999 hariç)"] = describe(pd_real)
t2.to_csv(P.RESULTS / "eda_numeric_summary.csv")
t2.round(3)

,n,mean,median,SD,Q1,Q3,IQR,min,max
age,3295.0,40.184,38.000,10.411,32.000,47.000,15.000,18.000,88.000
campaign,3295.0,2.533,2.000,2.577,1.000,3.000,2.000,1.000,35.000
pdays,3295.0,961.316,999.000,189.804,999.000,999.000,0.000,0.000,999.000
previous,3295.0,0.185,0.000,0.531,0.000,0.000,0.000,0.000,6.000
emp.var.rate,3295.0,0.090,1.100,1.573,-1.800,1.400,3.200,-3.400,1.400
cons.price.idx,3295.0,93.582,93.876,0.580,93.075,93.994,0.919,92.201,94.767
cons.conf.idx,3295.0,-40.404,-41.800,4.600,-42.700,-36.400,6.300,-50.800,-26.900
euribor3m,3295.0,3.630,4.857,1.736,1.334,4.961,3.627,0.635,5.045
nr.employed,3295.0,5166.610,5191.000,73.899,5099.100,5228.100,129.000,4963.600,5228.100
pdays (999 hariç),125.0,5.640,5.000,3.688,3.000,6.000,3.000,0.000,21.000


In [10]:
fig, axes = plt.subplots(3, 3, figsize=(11, 8.5))
for ax, c in zip(axes.ravel(), P.NUMERIC_COLS):
    s = num[c] if c != "pdays" else pd_real
    ax.hist(s, bins=30, color="#2b6cb0", alpha=.85)
    ax.set_title(c + (" (999 hariç)" if c == "pdays" else ""), fontsize=10)
fig.suptitle("Sayısal özelliklerin histogramları (development)")
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_eda_histograms.png", dpi=200); plt.close(fig)

fig, axes = plt.subplots(3, 3, figsize=(11, 8.5))
for ax, c in zip(axes.ravel(), P.NUMERIC_COLS):
    s = num[c] if c != "pdays" else pd_real
    data = [s[dev.loc[s.index, "y"] == 0], s[dev.loc[s.index, "y"] == 1]]
    ax.boxplot(data, tick_labels=["no", "yes"], widths=.6, flierprops={"markersize": 2})
    ax.set_title(c + (" (999 hariç)" if c == "pdays" else ""), fontsize=10)
fig.suptitle("Sayısal özelliklerin hedef sınıflarına göre boxplot'ları (development)")
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_eda_boxplots.png", dpi=200); plt.close(fig)

### 4.4 Kategorik özellikler: frekans tabloları, çubuk grafikler ve `unknown` kodları

In [11]:
freq_rows = []
for c in P.CATEGORICAL_COLS:
    vc = dev[c].value_counts()
    for k, v in vc.items():
        freq_rows.append({"feature": c, "category": k, "n": int(v), "pct": 100 * v / len(dev)})
freq = pd.DataFrame(freq_rows)
freq.to_csv(P.RESULTS / "eda_categorical_frequencies.csv", index=False)
unk = t1.loc[P.CATEGORICAL_COLS, ["n_unknown", "pct_unknown"]]
print(unk.round(2))
fig, axes = plt.subplots(5, 2, figsize=(11, 15))
for ax, c in zip(axes.ravel(), P.CATEGORICAL_COLS):
    vc = dev[c].value_counts()
    colors = ["#c05621" if k == "unknown" else "#2b6cb0" for k in vc.index]
    ax.barh(vc.index.astype(str)[::-1], vc.values[::-1], color=colors[::-1])
    ax.set_title(c, fontsize=10); ax.tick_params(labelsize=8)
fig.suptitle("Kategorik özelliklerin frekansları (turuncu = 'unknown')", y=0.995)
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_eda_categorical_bars.png", dpi=200); plt.close(fig)

             n_unknown  pct_unknown
feature                            
job                 29         0.88
marital              6         0.18
education          134         4.07
default            650        19.73
housing             83         2.52
loan                83         2.52
contact              0         0.00
month                0         0.00
day_of_week          0         0.00
poutcome             0         0.00


### 4.5 Korelasyon matrisi ve seçili özelliklerin hedefe göre betimsel karşılaştırması

In [12]:
corr = num.corr(method="pearson")
corr.to_csv(P.RESULTS / "eda_correlation_matrix.csv")
fig, ax = plt.subplots(figsize=(7.5, 6.3))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.values[i, j]) > .6 else "black")
fig.colorbar(im, ax=ax, shrink=.8); ax.set_title("Sayısal özellikler — Pearson korelasyonu (pdays ham, 999 dahil)")
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_eda_correlation.png", dpi=200); plt.close(fig)
print(corr.round(2))

                 age  campaign  pdays  previous  emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.employed
age             1.00     -0.02  -0.05      0.06         -0.03           -0.01           0.12      -0.02        -0.05
campaign       -0.02      1.00   0.06     -0.10          0.18            0.15           0.01       0.17         0.17
pdays          -0.05      0.06   1.00     -0.57          0.27            0.08          -0.10       0.30         0.37
previous        0.06     -0.10  -0.57      1.00         -0.42           -0.18          -0.03      -0.46        -0.51
emp.var.rate   -0.03      0.18   0.27     -0.42          1.00            0.75           0.17       0.97         0.90
cons.price.idx -0.01      0.15   0.08     -0.18          0.75            1.00           0.03       0.66         0.48
cons.conf.idx   0.12      0.01  -0.10     -0.03          0.17            0.03           1.00       0.26         0.09
euribor3m      -0.02      0.17   0.30     -0.46          0.97   

In [13]:
by_cls = num.assign(pdays_999=(num["pdays"] == 999).astype(int), y=dev["y"]).groupby("y").agg(["mean", "median"]).T
by_cls.columns = ["no (0)", "yes (1)"]
by_cls.to_csv(P.RESULTS / "eda_numeric_by_target.csv")
print(by_cls.round(3))
sel = ["poutcome", "contact", "default", "month"]
rate_rows = []
for c in sel:
    g = dev.groupby(c)["y"].agg(["count", "mean"])
    for k, r in g.iterrows():
        rate_rows.append({"feature": c, "category": k, "n": int(r["count"]), "positive_rate_pct": 100 * r["mean"]})
rates = pd.DataFrame(rate_rows)
rates.to_csv(P.RESULTS / "eda_positive_rate_by_category.csv", index=False)
rates.round(2)

                         no (0)   yes (1)
age            mean      39.959    42.014
               median    38.000    39.000
campaign       mean       2.603     1.964
               median     2.000     2.000
pdays          mean     983.773   778.798
               median   999.000   999.000
previous       mean       0.138     0.571
               median     0.000     0.000
emp.var.rate   mean       0.252    -1.230
               median     1.100    -1.800
cons.price.idx mean      93.605    93.400
               median    93.918    93.369
cons.conf.idx  mean     -40.497   -39.647
               median   -41.800   -40.300
euribor3m      mean       3.818     2.101
               median     4.857     1.260
nr.employed    mean    5175.865  5091.393
               median  5195.800  5076.200
pdays_999      mean       0.985     0.778
               median     1.000     1.000


,feature,category,n,positive_rate_pct
0,poutcome,failure,349,12.89
1,poutcome,nonexistent,2830,8.45
2,poutcome,success,116,66.38
3,contact,cellular,2110,14.36
4,contact,telephone,1185,4.89
5,default,no,2644,12.29
6,default,unknown,650,5.54
7,default,yes,1,0.00
8,month,apr,173,17.34
9,month,aug,526,10.46


### 4.6 Aykırı değerlerin işaretlenmesi (IQR ve %1–%99 yüzdelik)

In [14]:
out_rows = []
for c in P.NUMERIC_COLS:
    s = num[c] if c != "pdays" else pd_real
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    p1, p99 = s.quantile([.01, .99])
    out_rows.append({"feature": c + (" (999 hariç)" if c == "pdays" else ""),
                     "IQR_low": lo, "IQR_high": hi, "n_IQR_outlier": int(((s < lo) | (s > hi)).sum()),
                     "pct_IQR_outlier": 100 * ((s < lo) | (s > hi)).mean(),
                     "P1": p1, "P99": p99, "n_outside_P1_P99": int(((s < p1) | (s > p99)).sum())})
outl = pd.DataFrame(out_rows).set_index("feature")
outl.to_csv(P.RESULTS / "eda_outliers.csv")
outl.round(3)

,IQR_low,IQR_high,n_IQR_outlier,pct_IQR_outlier,P1,P99,n_outside_P1_P99
feature,,,,,,,
age,9.500,69.500,32,0.971,23.000,69.000,53
campaign,-2.000,6.000,192,5.827,1.000,13.000,32
pdays (999 hariç),-1.500,10.500,11,8.800,1.240,18.760,4
previous,0.000,0.000,465,14.112,0.000,2.000,32
emp.var.rate,-6.600,6.200,0,0.000,-3.400,1.400,0
cons.price.idx,91.697,95.372,0,0.000,92.201,94.465,32
cons.conf.idx,-52.150,-26.950,36,1.093,-50.000,-26.900,19
euribor3m,-4.107,10.402,0,0.000,0.677,4.968,47
nr.employed,4905.600,5421.600,0,0.000,4963.600,5228.100,0


**Aykırı değer yorumu.** `campaign` değişkeninde development satırlarının %5,8'i IQR sınırının (6) üzerindedir ve en büyük değer 35'tir. `previous` için Q1 = Q3 = 0 olduğundan IQR kuralı sıfır olmayan her değeri (%14,1) aykırı sayar; bu durumda IQR kuralı anlamlı bir aykırı tanımı vermez. `age` ve `cons.conf.idx` değişkenlerinde IQR dışında kalan oran %1 civarındadır. Makroekonomik değişkenlerde (`emp.var.rate`, `euribor3m`, `nr.employed`) IQR dışı değer yoktur. Bu değerlerin hiçbiri ölçüm hatası gibi görünmemektedir; hepsi olası müşteri ve kampanya durumlarıdır. `pdays = 999` aykırı değer değil, "daha önce temas yok" anlamındaki özel koddur ve silinmez.

**Öneri.** Hiçbir satır silinmeyecektir. Aykırı değer yönetimi seçeneği olarak sayısal sütunlarda, ilgili eğitim katının fit alt kümesinden öğrenilen %1 ve %99 yüzdeliklerine göre kırpma (clipping) kullanılacaktır. Bu seçenek otomatik aramanın arama uzayında yer alır ("yok" veya "eğitim katında öğrenilmiş kırpma"). Ayrıca M1 üzerinde yalnızca kırpmayı değiştiren bir ek analiz yapılacaktır; bu ek analiz seçim için kullanılmaz.

**Sabit ön işleme kuralları (model sonuçlarına bakılmadan belirlenmiştir).**
1. `pdays = 999` değerleri için `pdays_no_prev_contact` adında ikili bir gösterge sütunu eklenir. `pdays` içindeki 999 değerleri eksik (NaN) olarak işaretlenir ve her eğitim katında o katın fit alt kümesinden öğrenilen medyanla doldurulur. Böylece "temas yok" bilgisi korunur ve 999 değeri ölçeklemeyi bozmaz.
2. Kategorik `unknown` değerleri ana akışta ayrı bir kategori olarak tutulur. Bu, ödevin "unknown otomatik olarak NaN kabul edilmez" kuralının doğrudan uygulamasıdır ve hedefle ilişkiye bakılarak seçilmemiştir. Alternatif olan "unknown → NaN → fit alt kümesinin en sık sınıfı" yaklaşımı ek analizde karşılaştırılır.
3. Hedefle ilişkiye dayalı özellik eleme veya denetimli özellik seçimi yapılmaz. Birbiriyle çok yüksek korelasyonlu makro değişkenler (`emp.var.rate`–`euribor3m` r = 0,97; `euribor3m`–`nr.employed` r = 0,94) de bu nedenle veride bırakılır.

## 5. Deney hipotezleri (nested CV başlamadan önce yazılmıştır)

Aşağıdaki hipotezler yalnızca development EDA bulgularına dayanır. Hiçbir model sonucu görülmeden yazılmış ve bu commit ile kayıt altına alınmıştır.

**H1 — Sınıf dengesizliği ve metrik seçimi.** Development kümesinde pozitif sınıf oranı %10,96'dır. Her örneğe "no" diyen bir model %89 civarında doğruluk (accuracy) elde eder. Bu nedenle accuracy model karşılaştırmasında yanıltıcı olacaktır. DummyClassifier'ın AP değeri pozitif oranına (yaklaşık 0,11) eşit olacak, MLP ve Logistic Regression ise bunun belirgin biçimde üstünde AP verecektir. Ayrıca 0,50 eşiğinde bütün modellerin duyarlılığı (sensitivity) düşük kalacaktır (0,50'nin altında).

**H2 — Optimizer etkisi (M1, M2, M3).** Aynı öğrenme oranı (0,001) ile SGD + Momentum (M3), AdamW (M1) ve Adam'a (M2) göre daha yavaş yakınsayacaktır. Bu durum M3'te daha yüksek en iyi epoch değeri veya 150 epoch sınırına daha sık ulaşma olarak görülecek ve M3'ün iç CV ortalama AP değeri M1'den düşük olacaktır. Weight decay küçük (1e-4) olduğu için Adam ile AdamW arasındaki AP farkı küçük olacaktır.

**H3 — Ölçekleme etkisi (M1 ve M4).** Sayısal özelliklerin ölçekleri çok farklıdır (`nr.employed` yaklaşık 5.000, `cons.conf.idx` negatif) ve `campaign` gibi uzun kuyruklu değişkenler vardır. MinMaxScaler uzun kuyruklu değişkenlerde değerlerin çoğunu dar bir aralığa sıkıştırır. Bu nedenle StandardScaler (M1) en az MinMaxScaler (M4) kadar yüksek iç CV AP verecektir; farkın küçük olması beklenir.

**H4 — Mimari etkisi (M1 ve M5).** İç CV'de bir fit alt kümesinde yaklaşık 1.245 satır ve bunların yaklaşık 136'sı pozitif örnektir. Bu veri boyutunda (64, 32) mimarisi (M5) daha fazla parametreye sahip olduğu hâlde (32,) mimarisine (M1) göre AP artışı sağlamayacak; daha erken aşırı öğrenme nedeniyle en iyi epoch'u daha erken olacaktır.

**H5 — Learning rate scheduler etkisi (M1 ve M6).** Scheduler'ı kapatmak (M6) iç CV AP değerini anlamlı biçimde değiştirmeyecektir, çünkü erken durdurma her iki durumda da aktiftir. Scheduler açıkken öğrenme oranı düştüğü için eğitim daha geç durabilir.